# 6 - Uploading a local Data file with the SEEK API

In notebook 5 we uploaded a *Model* whose files we had fetched from the FAIRDOMHub. In this
notebook we upload a **Data file** from your own computer, and we pick a *large* one, so that the
techniques for big uploads (streaming, progress bar, checksums) become visible.

**Bring your own file.** The example used to write this notebook was `DMC_BGA22_4_N_75MB.tar.gz`,
a 140 MB archive with two FASTQ files. It is **not part of the repository**, because GitHub does not
accept files over 100 MB. Copy any file you like into the `06_Upload_Datafile` folder next to this
notebook, ideally something between 50 MB and a few hundred MB so the progress bar has something to
show, and enter its name in section 6.7. A large ZIP, a sequencing run, a microscopy stack, a video:
anything works, the content does not matter for the exercise.

**What you will learn**

* how to authenticate with an **API token** instead of a username/password (basic authentication)
* how to build the JSON payload that describes a Data file
* the three steps SEEK uses to upload a file through the API:

| Step | HTTP call | What happens |
|------|-----------|--------------|
| 1. Register | `POST /data_files` | SEEK creates the Data file *metadata* and an **empty content blob** (a placeholder that only knows the filename and mime type) |
| 2. Upload | `PUT /data_files/{id}/content_blobs/{blob_id}` | the raw bytes of the file are sent into that placeholder |
| 3. Verify | `GET /data_files/{id}` | SEEK now reports the size and checksums, which we compare against the local file |

Every asset that holds files (`data_files`, `models`, `sops`, `documents`, `presentations`, ...)
follows exactly the same pattern, so once you can do it for a Data file you can do it for all of them.

## 6.1 Prerequisites

1. **A SEEK instance you may write to.** During the workshop this is the training instance; the URL
   is set in section 6.3. Do *not* use https://fairdomhub.org for exercises.
2. **Membership in a project.** Every asset in SEEK must belong to at least one project and you can
   only add assets to projects you are a member of.
3. **An API token.** Create one in the web interface:

   *Click your name (top right)* → **My profile** → **Actions** → **API tokens** → **New API token**

   Give it a title (e.g. `api-workshop`) and copy the token immediately. SEEK shows the token **only
   once**; if you lose it, delete it and create a new one.

### Why an API token instead of basic authentication?

The older notebooks (and many SEEK examples on the web) use *basic authentication*, i.e. they send your
username and password with every request. A token is the better choice:

| | Basic authentication | API token |
|---|---|---|
| What is sent | your real password, on every request | a random string that only grants API access |
| If it leaks | the attacker owns your account | you delete that one token in your profile |
| Revocation | you must change your password | delete the token, everything else keeps working |
| Several scripts / machines | all share the same password | one token per script, each can be revoked separately |
| Accounts that log in via LDAP / SSO / ORCID | often no password at all, so it does not work | works |

The token is sent in the `Authorization` HTTP header in the form `Token <your token>`.

## 6.2 Import the libraries

* **requests** makes the HTTP calls. It is still the standard HTTP library for Python (it is
  actively maintained; run `pip install -U requests` if yours is older than 2.32).
* **json** pretty-prints the responses SEEK sends back.
* **os** and **getpass** let us read the token without ever writing it into the notebook.
* **pathlib** and **mimetypes** look at the local file (name, size, mime type).
* **urllib.parse** splits URLs into parts; we need it to fix links that SEEK generates (see 6.10).
* **hashlib** computes MD5/SHA1 checksums so we can prove the upload was not corrupted.
* **tqdm** draws a progress bar; useful for a 140 MB upload (`pip install tqdm`).

In [ ]:
import os
import json
import getpass
import hashlib
import mimetypes
from pathlib import Path
from urllib.parse import urlparse

import requests
from tqdm.auto import tqdm

## 6.3 Configuration: where is SEEK and who are you?

The notebook needs two settings:

* **`base_url`** - the address of the SEEK server *without* a trailing slash. Every API route is built
  by appending to it, e.g. `base_url + "/data_files"`.
* **`api_token`** - the token you created in section 6.1.

**Never paste a token (or a password) into a notebook cell.** Notebooks get committed to Git, shared by
e-mail and shown on projectors. Instead, keep the secret in a small file that stays on your machine.

### The config file `seek_config.json`

In the **root folder of this repository** there is a file called `seek_config.json`. Open it in any text
editor and fill in your values:

```json
{
    "base_url": "http://localhost:3000",
    "api_token": "paste-your-token-here"
}
```

The file is listed in `.gitignore`, so Git will never commit it. If it is missing, copy
`seek_config.example.json` to `seek_config.json` first.

### Order of precedence

The cell below looks for the settings in this order and uses the first one it finds:

1. the environment variables `SEEK_BASE_URL` / `SEEK_API_TOKEN` (useful for scripts and automation)
2. `seek_config.json` (the normal case during the workshop)
3. if there is still no token, `getpass` opens a hidden input field under the cell

This is the usual pattern for secrets in Python projects: code is committed, configuration is not.

In [ ]:
def load_config():
    '''Return {"base_url": ..., "api_token": ...} from seek_config.json, or {} if the file is missing.'''
    for candidate in (Path("seek_config.json"), Path("../seek_config.json")):
        if candidate.is_file():
            with open(candidate) as f:
                cfg = json.load(f)
            print("Read settings from", candidate.name)
            return cfg
    print("No seek_config.json found (looked in this folder and its parent)")
    return {}


config = load_config()

base_url = (os.environ.get("SEEK_BASE_URL") or config.get("base_url") or "http://localhost:3000").rstrip("/")

api_token = os.environ.get("SEEK_API_TOKEN") or config.get("api_token")
if not api_token or api_token == "paste-your-token-here":
    api_token = getpass.getpass("Paste your SEEK API token: ")

print("Using SEEK instance:", base_url)
print("Token loaded, length:", len(api_token))   # we only print the length, never the token itself

## 6.4 Authenticate with the API token

We create a `requests.Session`. A session remembers headers (and cookies, and the TCP connection), so we
configure the authentication **once** instead of repeating it in every call:

* `Accept: application/json` - we want JSON back, not HTML
* `Content-Type: application/json` - what we send in the body is JSON (we will override this for the
  binary upload)
* `Authorization: Token <token>` - identifies you to SEEK

For comparison, basic authentication would have been `session.auth = ("username", "password")`.

In [ ]:
session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Token {api_token}",
})

print("Session ready for", base_url)

## 6.5 Two small helper functions

`json_for_resource` is the same helper as in the previous notebooks, but it now goes through the
authenticated `session`, so it can also read resources that are not public.

`check_response` prints SEEK's error message when something goes wrong. SEEK returns errors as JSON,
for example `{"errors": [{"title": "Unauthorized", ...}]}`, and seeing that text is far more useful
than a bare status code. It then raises an exception, so a failed call stops the notebook instead of
silently continuing with broken data.

In [ ]:
def check_response(response, expected=(200, 201)):
    '''Print SEEK's error text (if any) and raise if the status code is not one we expect.'''
    if response.status_code not in expected:
        print(f"HTTP {response.status_code} for {response.request.method} {response.url}")
        try:
            print(json.dumps(response.json(), indent=2))
        except ValueError:                # body was not JSON (e.g. an HTML error page)
            print(response.text[:1000])
        response.raise_for_status()
        raise RuntimeError(f"Unexpected status code {response.status_code}")
    return response


def json_for_resource(resource_type, resource_id):
    '''GET {base_url}/{resource_type}/{resource_id} and return the parsed JSON.'''
    url = f"{base_url}/{resource_type}/{resource_id}"
    response = session.get(url)
    check_response(response)
    return response.json()

## 6.6 Check that the token works and pick the project

The Data file must be linked to a project. Set `containing_project_id` to the id of a project you are a
member of (open the project in the browser; the id is the number in the URL, e.g. `/projects/1`).

Fetching the project with the authenticated session is also a good smoke test for the token:

* **401 Unauthorized** - the token is wrong, was deleted, or the header is malformed
* **403 Forbidden** - the token is fine but you are not allowed to see this project
* **404 Not Found** - no project with this id

In [ ]:
containing_project_id = 1     # <-- change this to your project id

project = json_for_resource("projects", containing_project_id)

print("Project title :", project["data"]["attributes"]["title"])
print("Project URL   :", f"{base_url}/projects/{containing_project_id}")
print("Members       :", len(project["data"]["relationships"]["people"]["data"]))

## 6.7 Look at the local file

Before talking to SEEK we collect what we know about the file:

* **name** - becomes `original_filename` in SEEK, the name people will see when they download it
* **size** - lets us draw a progress bar and check the upload afterwards
* **mime type** - becomes `content_type`. SEEK uses it to decide how to display the file (e.g. show a
  spreadsheet as a table) and what to send in the `Content-Type` header when someone downloads it.

**Enter the name of your file** in the first line of the cell below. The file must be in the same
folder as this notebook.

Python's `mimetypes.guess_type` guesses the mime type from the extension and is right for most files
(`.csv`, `.pdf`, `.png`, `.zip`, ...). Two cases need help:

* for a `.tar.gz` (or `.fastq.gz`) it answers `application/x-tar` with a separate *encoding* `gzip`,
  which is not a single mime type; the accurate choice for a gzip file is `application/gzip`
* for an unknown extension it answers `None`

`application/octet-stream` ("unknown binary data") is always acceptable as a fallback.

In [ ]:
file_path = Path("DMC_BGA22_4_N_75MB.tar.gz")     # <-- put YOUR file name here (file must be in this folder)

if not file_path.is_file():
    raise FileNotFoundError(
        f"'{file_path}' is not in the folder {Path.cwd().name}. "
        "Copy the file you want to upload next to this notebook and set file_path to its name.")

file_size = file_path.stat().st_size
guessed_type, guessed_encoding = mimetypes.guess_type(file_path.name)

if guessed_encoding == "gzip":
    content_type = "application/gzip"
elif guessed_type:
    content_type = guessed_type
else:
    content_type = "application/octet-stream"

print("File name     :", file_path.name)
print("Size          :", f"{file_size:,} bytes  ({file_size / 1024**2:.1f} MiB)")
print("Guessed type  :", guessed_type, "| encoding:", guessed_encoding)
print("We will use   :", content_type)

## 6.8 Describe the Data file (the JSON payload)

The payload follows the [JSON:API](https://jsonapi.org/) layout you have seen when *reading* resources:
`data` → `type`, `attributes`, `relationships`. Only three things are mandatory:

* `attributes.title`
* `attributes.content_blobs` - a list with **one placeholder per file**; each placeholder only needs
  `original_filename` and `content_type`. This tells SEEK "a file will follow". (Data files currently
  hold one blob; Models may hold several.)
* `relationships.projects`

Everything else is optional. We add a `description`, some `tags`, and a `policy`.

**Sharing policy.** `policy.access` is the default access for everybody (including anonymous visitors):

| `access` | Meaning |
|---|---|
| `no_access` | private: only you and the people in `permissions` |
| `view` | others can see the metadata but not download the file |
| `download` | others can view and download |
| `edit` | others can also change the metadata and upload new versions |
| `manage` | others can also change the sharing policy and delete it |

`permissions` grants extra rights to specific people, projects or institutions. Below, everybody in the
project may edit while the rest of the world may only view.

The **assays** relationship is how a Data file becomes part of an ISA structure (Investigation → Study →
Assay). It is optional, so it is left commented out; the exercise at the end asks you to add it.

In [ ]:
data_file = {
    "data": {
        "type": "data_files",
        "attributes": {
            "title": f"{file_path.name} (API workshop upload)",      # <-- give your file a proper title
            "description": f"Test upload of {file_path.name} ({file_size / 1024**2:.1f} MiB) "
                           "during the SEEK API workshop. Replace this with a real description.",
            "tags": ["api-workshop"],
            "license": "CC-BY-4.0",
            "policy": {
                "access": "view",
                "permissions": [
                    {"resource": {"id": str(containing_project_id), "type": "projects"},
                     "access": "edit"}
                ],
            },
            # one placeholder per file that will be uploaded in step 2
            "content_blobs": [
                {"original_filename": file_path.name, "content_type": content_type}
            ],
        },
        "relationships": {
            "projects": {"data": [{"id": str(containing_project_id), "type": "projects"}]},
            # "assays": {"data": [{"id": "1", "type": "assays"}]},
        },
    }
}

print(json.dumps(data_file, indent=2))

## 6.9 Step 1 - Register the Data file (`POST /data_files`)

`session.post(..., json=data_file)` serialises the dictionary to JSON for us.

SEEK answers with the full JSON of the new Data file. The interesting part is
`attributes.content_blobs[0]`: it already has a `link` (the address where we must send the bytes) but
`size`, `md5sum` and `sha1sum` are still `null` because nothing has been uploaded yet.

The API documentation says the status should be **201 Created**; some SEEK versions return **200**,
so we accept both.

In [ ]:
response = session.post(f"{base_url}/data_files", json=data_file)
check_response(response, expected=(200, 201))

registered = response.json()
data_file_id = registered["data"]["id"]
blob = registered["data"]["attributes"]["content_blobs"][0]

print("Status code   :", response.status_code)
print("Data file id  :", data_file_id)
print("Data file URL :", f"{base_url}/data_files/{data_file_id}")
print("Content blob  :")
print(json.dumps(blob, indent=2))

## 6.10 Step 2 - Upload the bytes (`PUT .../content_blobs/{blob_id}`)

Now we send the file itself to the content blob route from the placeholder.

**Do not trust the host name in `link`.** SEEK builds every absolute URL it returns (`link`, the
`url` entries under `versions`, `meta.base_url`) from an *admin setting* called "site base URL", not
from the address you connected to. On a freshly installed or cloned instance that setting is often still
`http://localhost:3000`, so the link points to a server that does not exist from your machine. This is a
common trap when a SEEK is copied to a new host, put behind a reverse proxy, or exposed under a different
domain. The **path** part of the link (`/data_files/5/content_blobs/16`) is always correct, so we keep
the path and put our own `base_url` in front of it. `urlparse` does the splitting.

Then three details of the request itself matter:

* **`Content-Type: application/octet-stream`** tells SEEK the body is raw binary data. Without it the
  session's default `application/json` would be sent and SEEK would try to parse 140 MB of gzip as JSON
  and fail. Passing `headers=` to a single call overrides the session header for that call only.
* **Stream from disk.** Notebook 5 read the whole file into a `bytes` object. That is fine for an 80 kB
  model but not for large data: `requests` accepts an open file object as `data=` and streams it in
  chunks, so memory use stays small regardless of file size. It reads the file size itself and sets
  `Content-Length` accordingly.
* **Timeouts.** `timeout=(10, 600)` means: give up if the server does not *accept the connection* within
  10 s, or does not *answer* within 600 s after the upload finished (SEEK calculates checksums before
  answering, which takes a few seconds for big files).

`tqdm.wrapattr` wraps the file's `read()` method so every chunk `requests` reads advances the progress bar.

In [ ]:
def on_our_server(link):
    '''Replace the host in a URL returned by SEEK with our base_url, keeping only its path.'''
    return base_url + urlparse(link).path


blob_url = on_our_server(blob["link"])
print("Link from SEEK :", blob["link"])
print("Upload target  :", blob_url)

with open(file_path, "rb") as f, tqdm.wrapattr(
        f, "read", total=file_size, unit="B", unit_scale=True, unit_divisor=1024,
        desc=f"Uploading {file_path.name}") as stream:
    response = session.put(
        blob_url,
        data=stream,
        headers={"Content-Type": "application/octet-stream"},
        timeout=(10, 600),
    )

check_response(response, expected=(200, 201, 204))
print("Upload finished with status code", response.status_code)

## 6.11 Step 3 - Verify the upload

Fetch the Data file again. `size`, `md5sum` and `sha1sum` are now filled in by SEEK.

Checksums are the reliable way to know that what arrived on the server is byte-for-byte what we have on
disk; comparing sizes alone would not detect a corrupted transfer. We compute the same checksums locally
(reading the file in 1 MiB chunks so it never has to fit into memory) and compare.

In [ ]:
def local_checksums(path, chunk_size=1024 * 1024):
    '''Return (md5, sha1) hex digests of a file, reading it chunk by chunk.'''
    md5, sha1 = hashlib.md5(), hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            md5.update(chunk)
            sha1.update(chunk)
    return md5.hexdigest(), sha1.hexdigest()


local_md5, local_sha1 = local_checksums(file_path)

uploaded = json_for_resource("data_files", data_file_id)
remote_blob = uploaded["data"]["attributes"]["content_blobs"][0]

print(f"{'':10}{'local':42}{'SEEK':42}")
print(f"{'size':10}{file_size:<42}{remote_blob['size']:<42}")
print(f"{'md5':10}{local_md5:42}{remote_blob['md5sum']:42}")
print(f"{'sha1':10}{local_sha1:42}{remote_blob['sha1sum']:42}")

assert remote_blob["size"] == file_size, "size differs"
assert remote_blob["md5sum"] == local_md5, "MD5 differs"
assert remote_blob["sha1sum"] == local_sha1, "SHA1 differs"
print("\nUpload verified: the file on the server is identical to the local file.")

The Data file is now complete. Open it in the browser, and note the download route: as in notebook 3,
appending `/download` to the content blob link fetches the file (again with the host corrected).

In [ ]:
print("View in SEEK :", f"{base_url}/data_files/{data_file_id}")
print("Download     :", on_our_server(remote_blob["link"]) + "/download")

## 6.12 Optional - Update the metadata afterwards (`PATCH /data_files/{id}`)

Metadata and content are independent. You can change title, description, tags, policy, etc. at any time
with a `PATCH` request that contains only the attributes you want to change (plus the `id` and `type`).
The uploaded file is not touched.

In [ ]:
update = {
    "data": {
        "id": data_file_id,
        "type": "data_files",
        "attributes": {
            "description": uploaded["data"]["attributes"]["description"]
                           + "\n\nChecksums verified after upload via the API.",
            "tags": ["api-workshop", "verified"],
        },
    }
}

response = session.patch(f"{base_url}/data_files/{data_file_id}", json=update)
check_response(response)

print("Tags are now:", response.json()["data"]["attributes"]["tags"])

## 6.13 Exercises

1. **Your own file.** Point `file_path` at a file of your choice (a CSV, an image, a PDF). Which
   `content_type` should it get? Check the result in the browser: does SEEK render it?
2. **Attach to an assay.** Create an Assay in the web interface (or via the API), then uncomment the
   `assays` relationship in section 6.8 and re-run the registration. Where does the Data file appear now?
3. **Private upload.** Set `policy.access` to `no_access` and remove the project permission. Log out of
   the browser (or open a private window): can you still open the Data file URL?
4. **Break it on purpose.** Remove the `headers=` argument from the `PUT` in section 6.10 and run it
   again. Read the error SEEK sends back.
5. **Several files, one loop.** Write a loop that registers and uploads every file in a folder, reusing
   the three steps. Tip: `mimetypes.guess_type` covers most common file types.

## 6.14 Clean up (optional)

A training instance fills up quickly with 140 MB uploads. `DELETE /data_files/{id}` removes the Data file
together with its content. Set the flag to `True` to run it; it is off by default so you can still look at
the result in the browser.

In [ ]:
DELETE_AFTER_UPLOAD = False

if DELETE_AFTER_UPLOAD:
    response = session.delete(f"{base_url}/data_files/{data_file_id}")
    check_response(response, expected=(200, 204))
    print(f"Data file {data_file_id} deleted (status {response.status_code})")
else:
    print(f"Kept data file {data_file_id}. Set DELETE_AFTER_UPLOAD = True to remove it.")

## 6.15 Troubleshooting

| Symptom | Likely cause | What to do |
|---|---|---|
| `401 Unauthorized` on the first GET | token wrong or deleted; header not `Token <token>` | create a new token in *My profile → Actions → API tokens* |
| `403 Forbidden` on the POST | you are not a member of `containing_project_id` | pick a project you belong to |
| `422 Unprocessable Entity` on the POST | payload rejected (missing `title`, `content_blobs` or `projects`; unknown attribute) | read the `errors` printed by `check_response`, compare with section 6.8 |
| `413 Request Entity Too Large` on the PUT | the web server in front of SEEK (nginx/Apache) caps the body size | ask the SEEK admin to raise `client_max_body_size` |
| `ReadTimeout` on the PUT | slow network or SEEK still computing checksums | raise the second number in `timeout=(10, 600)` |
| checksums differ in 6.11 | interrupted or duplicated upload | repeat step 2 - a PUT to the same blob simply replaces the content |
| `size` stays `null` after the PUT | `Content-Type` was not `application/octet-stream` | see section 6.10 |
| `ConnectionError` on the PUT, links in responses say `localhost:3000` | the SEEK admin setting "site base URL" is wrong on that instance | keep the path only and prepend your `base_url` (section 6.10); tell the admin |

**Summary.** Uploading through the SEEK API is always *register metadata with a placeholder → PUT the bytes
→ verify checksums*, authenticated by a revocable API token sent as `Authorization: Token <token>`.